In [1]:
import os
import sys
from pathlib import Path

# Get the current notebook directory
notebook_dir = Path.cwd()
# Get parent directory (databricks folder)
parent_dir = notebook_dir.parent

# Change working directory to parent folder
os.chdir(parent_dir)

# Add parent directory to sys.path for imports
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

In [22]:
import os
from databricks.sdk.core import Config

# Method 1: Try to get token from environment variable first
DATABRICKS_TOKEN = os.getenv("DATABRICKS_TOKEN")

# Method 2: If not in environment, try to get from Databricks SDK Config
if not DATABRICKS_TOKEN:
    try:
        cfg = Config()
        # Try to authenticate and get token
        # This will work if you've run 'databricks auth login'
        auth_headers = cfg.authenticate()
        # If it's a dict with Authorization header, extract token
        if isinstance(auth_headers, dict) and "Authorization" in auth_headers:
            DATABRICKS_TOKEN = auth_headers["Authorization"].replace("Bearer ", "")
        else:
            # Try to get from config
            DATABRICKS_TOKEN = getattr(cfg, 'token', None)
    except Exception as e:
        print(f"⚠️ Could not get token from SDK: {e}")

# Method 3: If still no token, prompt user or raise error
if not DATABRICKS_TOKEN:
    raise ValueError(
        "DATABRICKS_TOKEN is required. Please either:\n"
        "1. Set environment variable: $env:DATABRICKS_TOKEN = 'your-token'\n"
        "2. Run 'databricks auth login' in terminal\n"
        "3. Or set it directly in this cell (not recommended for production)"
    )

# Set all required environment variables for MLflow
os.environ["DATABRICKS_HOST"] = "https://dbc-5eabeaeb-998c.cloud.databricks.com"
os.environ["DATABRICKS_TOKEN"] = DATABRICKS_TOKEN
os.environ["MLFLOW_TRACKING_URI"] = "databricks"
os.environ["MLFLOW_REGISTRY_URI"] = "databricks-uc"
os.environ["MLFLOW_USE_DATABRICKS_SDK_MODEL_ARTIFACTS_REPO_FOR_UC"] = "True"

print("✅ Databricks authentication configured for MLflow")
print(f"✅ Host: {os.environ['DATABRICKS_HOST']}")
print(f"✅ Token: {'*' * 20} (hidden)")

✅ Databricks authentication configured for MLflow
✅ Host: https://dbc-5eabeaeb-998c.cloud.databricks.com
✅ Token: ******************** (hidden)


In [3]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [23]:
import numpy as np
import pandas as pd
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import functions as F
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor
from xgboost.spark import SparkXGBClassifier
import os

import requests
import mlflow
import mlflow.pyfunc
from mlflow.models.signature import infer_signature
from mlflow.models import infer_signature
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np


# Read Data

In [5]:
spark_df = spark.table("workspace.default.xgb_training_panel")
pdf = spark_df.toPandas()

# Columns preprocessing

In [6]:
label_col = "units_sold"
categorical_cols = pdf.select_dtypes(include=["object"]).columns.tolist()
categorical_cols = [c for c in categorical_cols if c != label_col and c != "date"]
numeric_cols = [
    c for c in pdf.columns if c not in categorical_cols + [label_col, "date"]
]


In [7]:
categorical_cols

['product_id', 'channel']

In [8]:
numeric_cols

['avg_price',
 'promo_flag',
 'promo_spend',
 'macro_index',
 'seasonality',
 'channel_weight',
 'month',
 'quarter',
 'year',
 'units_sold_lag1',
 'units_sold_lag2',
 'units_sold_lag3',
 'units_sold_lag6',
 'units_sold_lag12',
 'rolling_mean_3',
 'rolling_mean_6',
 'rolling_std_6',
 'promo_rolling_sum_3',
 'avg_price_lag1',
 'avg_price_change_pct',
 'units_sold_diff',
 'units_sold_pct_change']

In [9]:
encoder = OrdinalEncoder()
original_pdf = pdf.copy()
pdf[categorical_cols] = encoder.fit_transform(pdf[categorical_cols])
category_maps = {
    col: {cat: idx for idx, cat in enumerate(cats)}
    for col, cats in zip(categorical_cols, encoder.categories_)
}

In [10]:
category_maps

{'product_id': {'JARDIANCE': 0, 'OFEV': 1},
 'channel': {'CASH_RETAIL': 0,
  'CHARGEBACKS_340B': 1,
  'MANAGED_CARE': 2,
  'MEDICAID': 3,
  'MEDICARE_PART_D': 4}}

In [11]:
X = pdf[categorical_cols + numeric_cols]
y = pdf[label_col]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Train model

In [13]:
model = XGBRegressor(
    tree_method="hist",
    enable_categorical=True,
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
)

model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

# Log model in airflow

In [14]:

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Users/pavelhym@gmail.com/xgboost-forecasting")



cat_mapping = {
    col: {cat: idx for idx, cat in enumerate(cats)}
    for col, cats in zip(categorical_cols, getattr(encoder, "categories_", []))
}

with mlflow.start_run() as run:
    model._estimator_type = "regressor"
    mlflow.log_params(model.get_params())

    # Evaluate model on test data
    y_pred = model.predict(X_test).clip(0)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Log metrics to MLflow
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    input_example = X_train.iloc[:5]
    signature = infer_signature(X_train, model.predict(X_train))

    booster = model.get_booster()
    mlflow.xgboost.log_model(
        xgb_model=booster,
        artifact_path="model",
        registered_model_name="xgboost_units",
        input_example=input_example,
        signature=signature,
    )

    if cat_mapping:
        mlflow.log_dict(cat_mapping, "categorical_mappings.json")

    run_id = run.info.run_id

print(f"MLflow run logged: {run_id}")
print(f"Test metrics -> RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.4f}")


d:\Documents\Bicocca\2Year\databricks\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
d:\Documents\Bicocca\2Year\databricks\.venv\Lib\site-packages\mlflow\xgboost\__init__.py:168: UserWarning: [20:52:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1575: Saving model in the UBJSON format 

🏃 View run enchanting-fox-220 at: https://dbc-5eabeaeb-998c.cloud.databricks.com/ml/experiments/3531510795245178/runs/755e37b33ba94e2f9c95cf5b4eef3787
🧪 View experiment at: https://dbc-5eabeaeb-998c.cloud.databricks.com/ml/experiments/3531510795245178
MLflow run logged: 755e37b33ba94e2f9c95cf5b4eef3787
Test metrics -> RMSE: 3808.79, MAE: 1885.70, R2: 0.9930


# Serve model

In [16]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedModelInput
from mlflow.tracking import MlflowClient
import time


In [17]:

# Initialize clients
w = WorkspaceClient()
mlflow_client = MlflowClient()

# Configuration
endpoint_name = "xgboost_units"
model_name = "workspace.default.xgboost_units"

print(f"🔍 Checking for model: {model_name}")

# Get the latest model version
try:
    versions = mlflow_client.search_model_versions(f"name='{model_name}'")
    if not versions:
        raise ValueError(f"Model '{model_name}' not found in registry")
    
    # Get the latest version
    latest_version = max(versions, key=lambda v: int(v.version))
    print(f"✅ Found latest model version: {latest_version.version}")
    print(f"   Model stage: {latest_version.current_stage}")
    print(f"   Model URI: models:/{model_name}/{latest_version.version}")
    
except Exception as e:
    print(f"❌ Error getting model version: {e}")
    raise


🔍 Checking for model: workspace.default.xgboost_units
✅ Found latest model version: 6
   Model stage: None
   Model URI: models:/workspace.default.xgboost_units/6


In [18]:

# Check if endpoint exists
endpoint_exists = False
try:
    existing_endpoint = w.serving_endpoints.get(endpoint_name)
    endpoint_exists = True
    print(f"\n✅ Endpoint '{endpoint_name}' already exists")
    print(f"   Current state: {existing_endpoint.state.config_update}")
except Exception as e:
    if "does not exist" in str(e) or "RESOURCE_DOES_NOT_EXIST" in str(e):
        print(f"\nℹ️  Endpoint '{endpoint_name}' does not exist, will create new one")
        endpoint_exists = False
    else:
        raise



✅ Endpoint 'xgboost_units' already exists
   Current state: EndpointStateConfigUpdate.NOT_UPDATING


In [19]:

# Create or update endpoint
if endpoint_exists:
    # Update existing endpoint
    print(f"\n🔄 Updating endpoint '{endpoint_name}' with model version {latest_version.version}...")
    
    w.serving_endpoints.update_config(
        name=endpoint_name,
        served_models=[
            ServedModelInput(
                model_name=model_name,
                model_version=latest_version.version,  # Use specific version
                workload_size="Small",  # Options: "Small", "Medium", "Large"
                scale_to_zero_enabled=True,  # Scale to zero when not in use
                environment_vars={}  # Add any env vars if needed
            )
        ]
    )
    print(f"✅ Endpoint update initiated")
    
else:
    # Create new endpoint
    print(f"\n🆕 Creating new endpoint '{endpoint_name}' with model version {latest_version.version}...")
    
    w.serving_endpoints.create(
        name=endpoint_name,
        config=EndpointCoreConfigInput(
            served_models=[
                ServedModelInput(
                    model_name=model_name,
                    model_version=latest_version.version,
                    workload_size="Small",
                    scale_to_zero_enabled=True,
                    environment_vars={}
                )
            ]
        )
    )
    print(f"✅ Endpoint creation initiated")

# Wait for endpoint to be ready
print(f"\n⏳ Waiting for endpoint '{endpoint_name}' to be ready...")
max_wait_time = 600  # 10 minutes
start_time = time.time()

while time.time() - start_time < max_wait_time:
    try:
        endpoint = w.serving_endpoints.get(endpoint_name)
        state = endpoint.state.config_update
        
        if state == "NOT_UPDATING":
            # Check if endpoint is ready
            if hasattr(endpoint.state, 'ready') and endpoint.state.ready == "READY":
                print(f"✅ Endpoint '{endpoint_name}' is READY!")
                break
            else:
                print(f"⏳ Endpoint state: {state}, waiting...")
        else:
            print(f"⏳ Endpoint updating: {state}, waiting...")
        
        time.sleep(10)  # Wait 10 seconds before checking again
        
    except Exception as e:
        print(f"⚠️  Error checking endpoint status: {e}")
        time.sleep(10)

# Get final endpoint info
try:
    endpoint = w.serving_endpoints.get(endpoint_name)
    print(f"\n📊 Endpoint Information:")
    print(f"   Name: {endpoint.name}")
    print(f"   State: {endpoint.state.config_update}")
    print(f"   Endpoint URL: https://dbc-5eabeaeb-998c.cloud.databricks.com/serving-endpoints/{endpoint_name}/invocations")
    
    if hasattr(endpoint, 'served_models') and endpoint.served_models:
        for i, served_model in enumerate(endpoint.served_models):
            print(f"   Model {i+1}: {served_model.model_name} (version {served_model.model_version})")
            print(f"   Workload size: {served_model.workload_size}")
            
except Exception as e:
    print(f"⚠️  Could not get final endpoint info: {e}")

print(f"\n🎉 Endpoint '{endpoint_name}' is ready to serve predictions!")
print(f"💡 Your Streamlit app can now use this endpoint via XGB_ENDPOINT_NAME='{endpoint_name}'")


🔄 Updating endpoint 'xgboost_units' with model version 6...
✅ Endpoint update initiated

⏳ Waiting for endpoint 'xgboost_units' to be ready...
⏳ Endpoint updating: EndpointStateConfigUpdate.IN_PROGRESS, waiting...
⏳ Endpoint updating: EndpointStateConfigUpdate.IN_PROGRESS, waiting...
⏳ Endpoint updating: EndpointStateConfigUpdate.IN_PROGRESS, waiting...
⏳ Endpoint updating: EndpointStateConfigUpdate.IN_PROGRESS, waiting...
⏳ Endpoint updating: EndpointStateConfigUpdate.IN_PROGRESS, waiting...
⏳ Endpoint updating: EndpointStateConfigUpdate.IN_PROGRESS, waiting...
⏳ Endpoint updating: EndpointStateConfigUpdate.IN_PROGRESS, waiting...
⏳ Endpoint updating: EndpointStateConfigUpdate.IN_PROGRESS, waiting...
⏳ Endpoint updating: EndpointStateConfigUpdate.IN_PROGRESS, waiting...
⏳ Endpoint updating: EndpointStateConfigUpdate.IN_PROGRESS, waiting...
⏳ Endpoint updating: EndpointStateConfigUpdate.IN_PROGRESS, waiting...
⏳ Endpoint updating: EndpointStateConfigUpdate.IN_PROGRESS, waiting...
⏳ En

# Calling endpoint

In [20]:


workspace = os.getenv(
    "DATABRICKS_HOST", "https://dbc-5eabeaeb-998c.cloud.databricks.com"
)
url = f"{workspace}/serving-endpoints/xgboost_units/invocations"


In [24]:
payload = {
    "dataframe_records": [
        {
            "product_id": 0.0,
            "channel": 0.0,
            "avg_price": 220.0,
            "promo_flag": 0,
            "promo_spend": 0.0,
            "macro_index": 100.0,
            "seasonality": 1.0,
            "channel_weight": 0.2,
            "month": 12,
            "quarter": 4,
            "year": 2025,
            "units_sold_lag1": 1200.0,
            "units_sold_lag2": 1180.0,
            "units_sold_lag3": 1150.0,
            "units_sold_lag6": 1100.0,
            "units_sold_lag12": 1050.0,
            "rolling_mean_3": 1176.7,
            "rolling_mean_6": 1125.0,
            "rolling_std_6": 45.2,
            "promo_rolling_sum_3": 0,
            "avg_price_lag1": 215.0,
            "avg_price_change_pct": 0.01,
            "units_sold_diff": 20.0,
            "units_sold_pct_change": 0.017,
        }
    ]
}

resp = requests.post(
    url,
    headers={"Authorization": f"Bearer {DATABRICKS_TOKEN}", "Content-Type": "application/json"},
    json=payload,
    timeout=120,
)

print(resp.status_code)
print(resp.json())

200
{'predictions': [1099.635498046875]}
